In [ ]:
"""Train the COF detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.cof import COF
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
cof = COF(
    contamination=config.CONTAMINATION,
    n_neighbors=config.N_NEIGHBORS,
    method=config.METHOD,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
cof.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = cof.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    cof.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="cof",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)
